In [ ]:
# Step 1: Parameters and file paths (Muravyev, Pearson, and Pollet, 2025, JF, p. 3683)
import pathlib

import numpy as np
import pandas as pd
from scipy import stats

HIGH_SIR_IO_CUTOFF = 0.18
SENSITIVITY_CUTOFF_LOW = 0.15
SENSITIVITY_CUTOFF_HIGH = 0.20

MIN_STOCK_PRICE = 1.0
SMALL_CAP_CUTOFF_MILLIONS = 50.0

# Mandatory SEC Form 13F XML filings begin in 2013Q2 (2013-06-30)
MIN_13F_REPORT_DATE = "2013-06-30"

# "previous_quarter" (MPP p. 3683 benchmark) or "strict_45d_lag"
IO_TIMING_RULE = "previous_quarter"
CAP_IO_AT_ONE = False
NEWEY_WEST_LAGS = 6

PROCESSED_DIR = pathlib.Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEC_13F_IO_FILE = PROCESSED_DIR / "sec_13f_quarterly_io.csv.gz"
SHORT_INTEREST_FILE = PROCESSED_DIR / "wrds_monthly_short_interest.csv.gz"
CRSP_MONTHLY_FILE = PROCESSED_DIR / "wrds_crsp_monthly.csv.gz"
OUTPUT_PROXY_PANEL_FILE = PROCESSED_DIR / "mpp2025_short_sale_cost_proxy.csv.gz"

In [ ]:
# Step 2: Construct monthly SIR_IO proxy panel
def compute_reference_quarter_end(month_end: pd.Series, timing_rule: str) -> pd.Series:
    prev_qtr_end = month_end - pd.offsets.QuarterEnd(1)
    if timing_rule == "previous_quarter":
        return prev_qtr_end
    if timing_rule == "strict_45d_lag":
        two_qtrs_ago = month_end - pd.offsets.QuarterEnd(2)
        is_first_month_of_qtr = month_end.dt.month.isin([1, 4, 7, 10])
        return prev_qtr_end.mask(is_first_month_of_qtr, two_qtrs_ago)
    raise ValueError(f"Unsupported IO_TIMING_RULE: {timing_rule}")


def construct_mpp2025_proxy_panel(
    quarterly_io: pd.DataFrame,
    monthly_si: pd.DataFrame,
    crsp_monthly: pd.DataFrame,
    timing_rule: str,
    cap_io_at_one: bool,
) -> pd.DataFrame:
    # 1. Standardize input dates and numeric types
    io_df = quarterly_io.copy()
    io_df["rdate"] = pd.to_datetime(io_df["rdate"]) + pd.offsets.MonthEnd(0)
    io_df["cusip8"] = io_df["cusip8"].astype(str).str.strip().str.upper().str[:8]
    io_df["inst_shares"] = pd.to_numeric(io_df["inst_shares"], errors="coerce")
    io_df["num_inst_owners"] = pd.to_numeric(io_df["num_inst_owners"], errors="coerce")

    si_df = monthly_si.copy()
    si_df["month_end"] = pd.to_datetime(si_df["month_end"]) + pd.offsets.MonthEnd(0)
    si_df["permno"] = pd.to_numeric(si_df["permno"], errors="coerce").astype("int64")
    si_df["shortint"] = pd.to_numeric(si_df["shortint"], errors="coerce")

    crsp = crsp_monthly.copy()
    crsp["month_end"] = pd.to_datetime(crsp["month_end"]) + pd.offsets.MonthEnd(0)
    crsp["permno"] = pd.to_numeric(crsp["permno"], errors="coerce").astype("int64")
    crsp["cusip8"] = crsp["cusip8"].fillna("").astype(str).str.strip().str.upper().str[:8]
    crsp["shares_out"] = pd.to_numeric(crsp["shares_out"], errors="coerce")
    crsp["prc_abs"] = pd.to_numeric(crsp["prc_abs"], errors="coerce")
    crsp["ret"] = pd.to_numeric(crsp["ret"], errors="coerce")
    crsp["mktcap_millions"] = pd.to_numeric(crsp["mktcap_millions"], errors="coerce")

    # 2. Match quarterly 13F CUSIP holdings to CRSP permno at each quarter-end
    is_qtr_end = crsp["month_end"].dt.month.isin([3, 6, 9, 12]) & (crsp["cusip8"].str.len() == 8)
    crsp_qtr = crsp.loc[is_qtr_end, ["permno", "month_end", "cusip8", "shares_out"]].rename(
        columns={"month_end": "rdate", "shares_out": "qtr_shares_out"}
    )
    permno_qtr_io = pd.merge(
        crsp_qtr,
        io_df[["rdate", "cusip8", "inst_shares", "num_inst_owners"]],
        on=["rdate", "cusip8"],
        how="inner",
    )
    permno_qtr_io = permno_qtr_io.groupby(["permno", "rdate"], as_index=False).agg(
        inst_shares_qtr=("inst_shares", "sum"),
        num_inst_owners_qtr=("num_inst_owners", "max"),
        qtr_shares_out=("qtr_shares_out", "first"),
    )
    permno_qtr_io["io_qtr"] = permno_qtr_io["inst_shares_qtr"] / permno_qtr_io["qtr_shares_out"]
    if cap_io_at_one:
        permno_qtr_io["io_qtr"] = permno_qtr_io["io_qtr"].clip(upper=1.0)

    # 3. Merge monthly short interest and lagged quarter-end IO onto CRSP by permno
    panel = pd.merge(
        crsp,
        si_df[["permno", "month_end", "shortint"]],
        on=["permno", "month_end"],
        how="left",
    )
    panel["ref_qtr_end"] = compute_reference_quarter_end(panel["month_end"], timing_rule)

    lagged_io = permno_qtr_io[
        ["permno", "rdate", "io_qtr", "inst_shares_qtr", "num_inst_owners_qtr"]
    ].rename(
        columns={
            "rdate": "ref_qtr_end",
            "io_qtr": "io_prev_qtr",
            "inst_shares_qtr": "inst_shares_prev_qtr",
            "num_inst_owners_qtr": "num_inst_owners_prev_qtr",
        }
    )
    panel = pd.merge(panel, lagged_io, on=["permno", "ref_qtr_end"], how="left")

    # 4. Compute SIR_t, SIR_IO_t = SIR_t / IO_{q-1}, and high-short-cost indicators
    panel["sir"] = panel["shortint"] / panel["shares_out"]
    valid_proxy = panel["sir"].notna() & panel["io_prev_qtr"].notna() & (panel["io_prev_qtr"] > 0)
    panel["sir_io"] = np.where(valid_proxy, panel["sir"] / panel["io_prev_qtr"], np.nan)

    for col_name, cutoff in [
        ("high_short_cost_18pct", HIGH_SIR_IO_CUTOFF),
        ("high_short_cost_15pct", SENSITIVITY_CUTOFF_LOW),
        ("high_short_cost_20pct", SENSITIVITY_CUTOFF_HIGH),
    ]:
        flag = pd.Series(pd.NA, index=panel.index, dtype="Int64")
        has_proxy = panel["sir_io"].notna()
        flag.loc[has_proxy] = (panel.loc[has_proxy, "sir_io"] > cutoff).astype("int64")
        panel[col_name] = flag

    # 5. Align next-month return (t+1) and MPP (2025) sample eligibility flags
    panel = panel.sort_values(["permno", "month_end"]).reset_index(drop=True)
    expected_next_month = panel["month_end"] + pd.DateOffset(months=1) + pd.offsets.MonthEnd(0)
    actual_next_month = panel.groupby("permno")["month_end"].shift(-1)
    next_ret = panel.groupby("permno")["ret"].shift(-1)
    panel["ret_lead1"] = np.where(actual_next_month == expected_next_month, next_ret, np.nan)

    panel["is_common_major_exch"] = panel["common_stock"].fillna(False) & panel[
        "major_exchange"
    ].fillna(False)
    panel["price_ge_1"] = panel["prc_abs"] >= MIN_STOCK_PRICE
    panel["mktcap_ge_50m"] = panel["mktcap_millions"] >= SMALL_CAP_CUTOFF_MILLIONS
    panel["mpp_eligible_sample"] = (
        panel["is_common_major_exch"] & panel["price_ge_1"] & panel["sir_io"].notna()
    )

    panel["sir_io_pct_rank"] = np.nan
    elig_mask = panel["mpp_eligible_sample"]
    panel.loc[elig_mask, "sir_io_pct_rank"] = (
        panel.loc[elig_mask].groupby("month_end")["sir_io"].rank(pct=True)
    )
    return panel

In [ ]:
# Step 3: Summary statistics and MPP (2025) Table IX portfolio exclusion functions
def newey_west_mean_tstat(series: pd.Series, max_lags: int) -> tuple[float, float, float]:
    clean_vals = series.dropna().to_numpy(dtype="float64")
    n_obs = len(clean_vals)
    if n_obs < 3:
        return float("nan"), float("nan"), float("nan")

    mean_val = float(np.mean(clean_vals))
    demeaned = clean_vals - mean_val
    nw_var = float(np.dot(demeaned, demeaned)) / n_obs

    effective_lags = min(max_lags, n_obs - 1)
    for lag in range(1, effective_lags + 1):
        weight = 1.0 - lag / (effective_lags + 1.0)
        gamma_lag = float(np.dot(demeaned[lag:], demeaned[:-lag])) / n_obs
        nw_var += 2.0 * weight * gamma_lag

    var_of_mean = max(nw_var, 1e-16) / n_obs
    t_stat = mean_val / float(np.sqrt(var_of_mean))
    p_val = float(2.0 * stats.t.sf(abs(t_stat), df=n_obs - 1))
    return mean_val, t_stat, p_val


def compute_monthly_portfolio_return(sub: pd.DataFrame, weighting: str) -> pd.Series:
    if weighting == "ew":
        return sub.groupby("month_end")["ret_lead1"].mean()
    if weighting == "vw":
        weighted_ret = (sub["ret_lead1"] * sub["mktcap_millions"]).groupby(sub["month_end"]).sum()
        total_mktcap = sub["mktcap_millions"].groupby(sub["month_end"]).sum()
        return weighted_ret / total_mktcap
    raise ValueError(f"Unsupported weighting: {weighting}")


def evaluate_anomaly_mpp_table9(
    panel: pd.DataFrame,
    signal_col: str,
    high_cost_col: str = "high_short_cost_18pct",
    weighting: str = "ew",
    use_nyse_breakpoints: bool = False,
    nw_lags: int = 6,
) -> pd.DataFrame:
    req_mask = (
        panel["mpp_eligible_sample"]
        & panel[signal_col].notna()
        & panel["ret_lead1"].notna()
        & panel[high_cost_col].notna()
    )
    work = panel.loc[req_mask].copy()

    # Form deciles 1..10 on all eligible stocks first, then exclude SIR_IO > 18% without re-sorting
    decile_frames: list[pd.DataFrame] = []
    interior_q = np.linspace(0.1, 0.9, 9)
    for _, month_df in work.groupby("month_end"):
        if len(month_df) < 20:
            continue
        bp_vals = (
            month_df.loc[month_df["is_nyse"], signal_col]
            if use_nyse_breakpoints
            else month_df[signal_col]
        )
        if len(bp_vals) < 10:
            bp_vals = month_df[signal_col]

        cutoffs = np.unique(np.quantile(bp_vals.to_numpy(dtype="float64"), interior_q))
        month_copy = month_df.copy()
        if len(cutoffs) < 9:
            ranked = month_copy[signal_col].rank(method="first")
            month_copy["decile"] = pd.qcut(ranked, 10, labels=False) + 1
        else:
            bins: list[float] = [-float("inf")] + [float(x) for x in cutoffs] + [float("inf")]
            month_copy["decile"] = pd.cut(month_copy[signal_col], bins=bins, labels=False) + 1
        decile_frames.append(month_copy)

    if not decile_frames:
        raise RuntimeError("Insufficient monthly cross-sections to form 10 decile portfolios.")

    sorted_df = pd.concat(decile_frames, ignore_index=True)
    low_cost_df = sorted_df.loc[sorted_df[high_cost_col] == 0]

    all_d1 = compute_monthly_portfolio_return(sorted_df.loc[sorted_df["decile"] == 1], weighting)
    all_d10 = compute_monthly_portfolio_return(sorted_df.loc[sorted_df["decile"] == 10], weighting)
    all_ls = (all_d10 - all_d1).dropna()

    low_d1 = compute_monthly_portfolio_return(
        low_cost_df.loc[low_cost_df["decile"] == 1], weighting
    )
    low_d10 = compute_monthly_portfolio_return(
        low_cost_df.loc[low_cost_df["decile"] == 10], weighting
    )
    low_ls = (low_d10 - low_d1).dropna()

    diff_ls = (low_ls - all_ls).dropna()

    excl_overall = float((sorted_df[high_cost_col] == 1).mean()) * 100.0
    excl_d1 = float((sorted_df.loc[sorted_df["decile"] == 1, high_cost_col] == 1).mean()) * 100.0
    excl_d10 = float((sorted_df.loc[sorted_df["decile"] == 10, high_cost_col] == 1).mean()) * 100.0

    summary_records: list[dict[str, object]] = []
    for sample_label, d1_ser, d10_ser, ls_ser, excl_rate in [
        ("Panel A: All Eligible Stocks", all_d1, all_d10, all_ls, 0.0),
        (f"Panel D: Exclude High SIR/IO ({high_cost_col})", low_d1, low_d10, low_ls, excl_overall),
    ]:
        m_d1, t_d1, _ = newey_west_mean_tstat(d1_ser * 100.0, nw_lags)
        m_d10, t_d10, _ = newey_west_mean_tstat(d10_ser * 100.0, nw_lags)
        m_ls, t_ls, p_ls = newey_west_mean_tstat(ls_ser * 100.0, nw_lags)
        summary_records.append(
            {
                "Specification": sample_label,
                "Weighting": weighting.upper(),
                "Short_D1_pct": round(m_d1, 3),
                "Short_D1_t": round(t_d1, 2),
                "Long_D10_pct": round(m_d10, 3),
                "Long_D10_t": round(t_d10, 2),
                "Long_Short_pct": round(m_ls, 3),
                "Long_Short_t": round(t_ls, 2),
                "Long_Short_pval": round(p_ls, 4),
                "Excluded_All_pct": round(excl_rate, 2),
                "Excluded_D1_pct": round(excl_d1 if excl_rate > 0 else 0.0, 2),
                "Excluded_D10_pct": round(excl_d10 if excl_rate > 0 else 0.0, 2),
            }
        )

    m_diff, t_diff, p_diff = newey_west_mean_tstat(diff_ls * 100.0, nw_lags)
    summary_records.append(
        {
            "Specification": "Difference (Panel D minus Panel A L-S)",
            "Weighting": weighting.upper(),
            "Short_D1_pct": np.nan,
            "Short_D1_t": np.nan,
            "Long_D10_pct": np.nan,
            "Long_D10_t": np.nan,
            "Long_Short_pct": round(m_diff, 3),
            "Long_Short_t": round(t_diff, 2),
            "Long_Short_pval": round(p_diff, 4),
            "Excluded_All_pct": round(excl_overall, 2),
            "Excluded_D1_pct": round(excl_d1, 2),
            "Excluded_D10_pct": round(excl_d10, 2),
        }
    )
    return pd.DataFrame(summary_records)


def summarize_annual_slice(df_slice: pd.DataFrame, label: str) -> dict[str, object]:
    return {
        "Year": label,
        "Distinct_Stocks": int(df_slice["permno"].nunique()),
        "Stock_Months": int(len(df_slice)),
        "Mean_IO_pct": round(float(df_slice["io_win"].mean()), 2),
        "Mean_SIR_pct": round(float(df_slice["sir_win"].mean()), 2),
        "Median_SIR_IO_pct": round(float(df_slice["sir_io"].median() * 100.0), 2),
        "P90_SIR_IO_pct": round(float(df_slice["sir_io"].quantile(0.90) * 100.0), 2),
        "Excl_15pct": round(float((df_slice["high_short_cost_15pct"] == 1).mean() * 100.0), 2),
        "Excl_18pct": round(float((df_slice["high_short_cost_18pct"] == 1).mean() * 100.0), 2),
        "Excl_20pct": round(float((df_slice["high_short_cost_20pct"] == 1).mean() * 100.0), 2),
    }


def compute_summary_statistics_tables(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    elig_df = panel.loc[panel["mpp_eligible_sample"]].copy()
    elig_df["year"] = elig_df["month_end"].dt.year

    table1_specs: list[tuple[str, pd.Series, bool]] = [
        ("Market Capitalization ($M)", elig_df["mktcap_millions"], True),
        ("Stock Price ($)", elig_df["prc_abs"], True),
        ("Institutional Ownership IO_{q-1} (%)", elig_df["io_prev_qtr"] * 100.0, True),
        ("Number of 13F Owners N_{q-1}", elig_df["num_inst_owners_prev_qtr"], True),
        ("Short Interest Ratio SIR_t (%)", elig_df["sir"] * 100.0, True),
        ("Utilization Proxy SIR_IO_t (%)", elig_df["sir_io"] * 100.0, True),
        (
            "High Short-Sale Cost I(SIR_IO > 18%) (%)",
            elig_df["high_short_cost_18pct"].astype("float64") * 100.0,
            False,
        ),
        (
            "Cutoff Perturbation I(SIR_IO > 15%) (%)",
            elig_df["high_short_cost_15pct"].astype("float64") * 100.0,
            False,
        ),
        (
            "Cutoff Perturbation I(SIR_IO > 20%) (%)",
            elig_df["high_short_cost_20pct"].astype("float64") * 100.0,
            False,
        ),
    ]

    # Winsorize continuous Mean/SD at 1%/99%; keep exact quantiles and binary indicators
    pooled_rows: list[dict[str, object]] = []
    for label, raw_ser, do_winsor in table1_specs:
        clean_s = raw_ser.dropna()
        stat_s = (
            clean_s.clip(lower=float(clean_s.quantile(0.01)), upper=float(clean_s.quantile(0.99)))
            if do_winsor
            else clean_s
        )
        pooled_rows.append(
            {
                "Variable": label,
                "N": int(len(clean_s)),
                "Mean": round(float(stat_s.mean()), 2),
                "SD": round(float(stat_s.std(ddof=1)), 2),
                "P10": round(float(clean_s.quantile(0.10)), 2),
                "P25": round(float(clean_s.quantile(0.25)), 2),
                "Median": round(float(clean_s.quantile(0.50)), 2),
                "P75": round(float(clean_s.quantile(0.75)), 2),
                "P90": round(float(clean_s.quantile(0.90)), 2),
            }
        )

    io_pct = elig_df["io_prev_qtr"] * 100.0
    sir_pct = elig_df["sir"] * 100.0
    elig_df["io_win"] = io_pct.clip(
        lower=float(io_pct.quantile(0.01)), upper=float(io_pct.quantile(0.99))
    )
    elig_df["sir_win"] = sir_pct.clip(
        lower=float(sir_pct.quantile(0.01)), upper=float(sir_pct.quantile(0.99))
    )

    annual_rows: list[dict[str, object]] = []
    for yr_val, grp in elig_df.groupby("year"):
        annual_rows.append(summarize_annual_slice(grp, str(yr_val)))
    annual_rows.append(summarize_annual_slice(elig_df, "Full Sample"))

    return pd.DataFrame(pooled_rows), pd.DataFrame(annual_rows)

In [ ]:
# Step 4: Load processed datasets, build SIR_IO proxy panel, and print summary tables
quarterly_io = pd.read_csv(SEC_13F_IO_FILE, dtype={"cusip8": str}, low_memory=False)
quarterly_io["rdate"] = pd.to_datetime(quarterly_io["rdate"])
quarterly_io = quarterly_io.loc[
    quarterly_io["rdate"] >= pd.Timestamp(MIN_13F_REPORT_DATE)
].copy()
quarterly_io = quarterly_io.sort_values(["rdate", "cusip8"]).drop_duplicates(
    subset=["rdate", "cusip8"], keep="last"
)

monthly_si = pd.read_csv(SHORT_INTEREST_FILE, low_memory=False)
crsp_monthly = pd.read_csv(CRSP_MONTHLY_FILE, dtype={"cusip8": str}, low_memory=False)

proxy_panel = construct_mpp2025_proxy_panel(
    quarterly_io=quarterly_io,
    monthly_si=monthly_si,
    crsp_monthly=crsp_monthly,
    timing_rule=IO_TIMING_RULE,
    cap_io_at_one=CAP_IO_AT_ONE,
)
proxy_panel.to_csv(OUTPUT_PROXY_PANEL_FILE, index=False, compression="gzip")
print(f"Saved proxy panel ({len(proxy_panel):,} rows) -> {OUTPUT_PROXY_PANEL_FILE}")

elig = proxy_panel.loc[proxy_panel["mpp_eligible_sample"]]
excl_share = float((elig["high_short_cost_18pct"] == 1).mean()) * 100.0
print(f"Eligible stock-months: {len(elig):,}")
print(f"Excluded by SIR/IO > {HIGH_SIR_IO_CUTOFF:.0%}: {excl_share:.2f}%")

table1_pooled, table2_annual = compute_summary_statistics_tables(proxy_panel)
print("\n=== Table 1: Pooled Summary Statistics ===")
print(table1_pooled.to_string(index=False))
print("\n=== Table 2: Annual Evolution and Exclusion Rates ===")
print(table2_annual.to_string(index=False))

In [ ]:
# Step 5: Example MPP (2025) Table IX portfolio exclusion test (Low Return Volatility)
proxy_panel["ret_vol_12m"] = (
    proxy_panel.groupby("permno")["ret"]
    .rolling(12, min_periods=10)
    .std()
    .reset_index(level=0, drop=True)
)
proxy_panel["low_volatility"] = -proxy_panel["ret_vol_12m"]

ew_results = evaluate_anomaly_mpp_table9(
    panel=proxy_panel,
    signal_col="low_volatility",
    high_cost_col="high_short_cost_18pct",
    weighting="ew",
    use_nyse_breakpoints=False,
    nw_lags=NEWEY_WEST_LAGS,
)
vw_results = evaluate_anomaly_mpp_table9(
    panel=proxy_panel,
    signal_col="low_volatility",
    high_cost_col="high_short_cost_18pct",
    weighting="vw",
    use_nyse_breakpoints=True,
    nw_lags=NEWEY_WEST_LAGS,
)

table9_results = pd.concat([ew_results, vw_results], ignore_index=True)
print("=== Table 3: MPP (2025) Table IX Evaluation for 'low_volatility' ===")
print(table9_results.to_string(index=False))